# Kapitel 6 – Övningsuppgifter: Klustring

Mina svar på övningsuppgifterna till kapitel 6 ("Klustring") i boken *"Lär dig AI från grunden - Tillämpad maskininlärning med Python"* (Prgomet, Johnson, Solberg, Rundberg Streuli). Uppgifterna är hämtade från bokens GitHub-repo.

## Fråga 1 – Vad är klustring för något? Ge några exempel på tillämpningsområden

**Klustring** handlar om att dela in datapunkter i grupper, eller kluster, utifrån hur lika de är varandra – punkter som liknar varandra ska hamna i samma kluster. Det är ett exempel på icke-vägledd inlärning, för man har bara indata (X) och ingen facit-variabel (y) som styr indelningen. Modellen får alltså själv leta upp strukturen i datan.

Boken tar upp ett bra exempel med ett fotoalbum: en klustringsmodell kan gruppera bilder på samma person utan att egentligen veta vem personen är, till skillnad från klassificering där varje bild redan har en etikett med namnet. Den vanligaste klustringsmodellen är **K-Means** (avsnitt 6.2), som jag går igenom mer i nästa fråga.

Några exempel på var man faktiskt använder klustring i praktiken (avsnitt 6.1.1):

- Kundsegmentering: kunder som liknar varandra, t.ex. i köpbeteende, ålder och lön, grupperas ihop. Ett företag kan då rikta olika typer av marknadsföring mot olika kluster – ett kluster med mest pensionärer kanske svarar bättre på fysisk reklam medan ett kluster med mest studenter nås bättre via sociala medier (det här visas i figur 6.1, där kunder delas in i tre kluster utifrån lön och ålder).
- Anomalidetektion: klustring kan också användas för att hitta avvikelser. Enligt figur 6.2 finns det två varianter – antingen sticker en eller några observationer ut inom ett kluster, eller så avviker ett helt kluster från de andra (till exempel för att det bara innehåller ett fåtal observationer). Det här används bland annat för att hitta defekta produkter i tillverkning eller misstänkta banktransaktioner.
- Semi-vägledd inlärning: har man bara etiketter på en liten del av datan kan man klustra allt och sedan sätta etiketter baserat på vilket kluster respektive observation hamnar i (mer om det i avsnitt 6.3.2).

## Fråga 2 – Förklara översiktligt hur K-means fungerar. Använd figur 6.3 (sidan 238) och figur 6.4 (sidan 239) i din förklaring

**K-Means** är den klustringsmodellen man stöter på oftast, och principen bakom den är egentligen ganska enkel: varje punkt hamnar i det kluster vars centroid (klustrets "mittpunkt") ligger närmast. Själva träningen är iterativ – man startar med slumpmässigt placerade centroider, tilldelar varje observation till närmaste centroid, räknar ut medelvärdet av alla punkter i respektive kluster och flyttar centroiden dit, och upprepar detta (styrt av hyperparametern `max_iter`) tills lösningen slutar ändra sig. En sak man måste bestämma i förväg är antalet kluster, `n_clusters` – modellen räknar inte ut det åt en.

Grejen är att K-Means bara fungerar bra när tre saker stämmer: klustren är ungefär lika stora, de har liknande spridning, och de har en sfärisk form. Anledningen är att modellen enbart tittar på avstånd till centroiderna, vilket alltid ger runda/konvexa gränser oavsett hur datan faktiskt är formad.

Det här blir väldigt tydligt om man tittar på figur 6.3 och 6.4, som visar fyra simulerade scenarier. Figur 6.3 visar de sanna klustren (facit) i varje scenario, medan figur 6.4 visar vad K-Means faktiskt kommer fram till på samma data:

1. Tre kluster – ett ganska idealt fall, runda kluster av ungefär samma storlek och spridning.
2. Icke-sfäriska kluster – tre avlånga, diagonala "stråk" av punkter.
3. Kluster med olika spridning – ett av klustren är betydligt mer utspritt än de andra två.
4. Kluster av olika storlek – ett av klustren har mycket färre observationer än de andra.

När jag jämför bilderna ser jag att det första fallet (uppe till vänster i figur 6.4) egentligen bara går fel om man anger fel antal kluster vid träningen – datan i sig är ju snäll och sfärisk, problemet är att `n_clusters` är felställt. I det andra fallet (uppe till höger) är det formen som stökar till det: K-Means delar av de diagonala stråken med raka gränser istället för att följa deras form, så punkter från olika sanna kluster blandas ihop. Tredje fallet (nere till vänster) beror på att spridningen skiljer sig – det stora, utspridda klustret liksom äter sig in i grannklustret. Och i fjärde fallet (nere till höger) är det storleksskillnaden som stökar till det, eftersom K-Means har en tendens att skapa ungefär lika stora regioner och därför delar upp det lilla klustret tillsammans med ett större i närheten.

Som jag ser det visar de här två figurerna tillsammans samma poäng: K-Means letar i grunden bara efter närmaste centroid, vilket funkar utmärkt när klustren är runda, lika stora och lika spridda (som i fall 1, förutsatt att man valt rätt `n_clusters`), men ger missvisande resultat så fort någon av de förutsättningarna inte stämmer.

## Fråga 3 – (Resonemangsfråga) Hur kan man välja vilket antal kluster som ska användas för en K-means modell? Använd inertia, silhouette score och silhouette diagram i ditt svar

Att veta hur många kluster (`n_clusters`) man borde använda är svårare än det låter, särskilt för dataset med fler än två-tre variabler där man inte bara kan plotta datan och se det, som man kunde i figur 6.5 där fem kluster var ganska uppenbart. Boken tar upp tre verktyg som kan hjälpa till (avsnitt 6.2.2/6.2.3), och som jag ser det använder man helst alla tre tillsammans.

Det första är inertia, som är medelkvadratavståndet mellan varje observation och dess närmaste centroid – samma mått som K-Means själv försöker minimera under träningen (det används också för att välja den bästa av flera slumpmässiga initialiseringar, styrt av `n_init`). Problemet med att bara titta på inertia är att den alltid minskar när man lägger till fler kluster, så om man bara väljer modellen med lägst inertia hamnar man alltid på flest möjliga kluster, vilket inte säger särskilt mycket. Istället plottar man inertia mot antal kluster och letar efter en **armbåge** – punkten där kurvan slutar sjunka brant och börjar plana ut. Det kallas armbågsmetoden.

Silhouette score är lite mer sofistikerat. Det är medelvärdet av alla observationers siluettkoefficient:

$$SC = \frac{b-a}{\max(a,b)}$$

där $a$ är medelavståndet till andra punkter i samma kluster (*intra cluster distance*) och $b$ är medelavståndet till punkter i det närmaste andra klustret (*nearest cluster distance*). Koefficienten ligger mellan -1 och +1: nära +1 betyder att punkten passar bra i sitt eget kluster och ligger långt från andra kluster, nära 0 betyder att den ligger nära en gräns mellan kluster, och nära -1 tyder på att den förmodligen hamnat i fel kluster. Det som är bra med silhouette score jämfört med inertia är att den inte per automatik blir bättre av fler kluster, så man kan faktiskt använda den för att jämföra olika värden på `n_clusters` rakt av – även om skillnaden mellan två närliggande alternativ ibland är så liten att det ändå krävs lite eget omdöme.

Sen finns silhouette diagram, som ger en mer detaljerad bild genom att visa siluettkoefficienten för varje enskild observation, grupperad per kluster i en knivliknande form. Höjden på "kniven" visar hur många observationer klustret har, och bredden visar de sorterade siluettkoefficienterna – ju längre ut, desto bättre. En streckad linje markerar medelvärdet, alltså silhouette score. Det man vill se är att alla knivarna passerar den streckade linjen (annars har klustret för många observationer med sämre koefficient än snittet, vilket tyder på att de ligger nära ett annat kluster) och att knivarna är ungefär lika breda, eftersom K-Means som sagt presterar bäst när klustren är av liknande storlek (jämför fråga 2).

Så som jag ser det kompletterar de tre varandra: inertia ger en grov fingervisning via armbågen, silhouette score ger en mer exakt jämförelse mellan olika antal kluster, och silhouette diagram visar hur balanserade och välformade klustren faktiskt är. Men även med alla tre kvar krävs det i slutändan ett visst mått av eget omdöme (se även fråga 4).

## Fråga 4 – (Resonemangsfråga) Om du kollar på figur 6.10 på sidan 247, hur många kluster hade du valt och varför? Är det en "exakt vetenskap" att välja antalet kluster?

Figur 6.10 visar hur inertia förändras med antalet kluster, från 1 till 9, för samma dataset som i figur 6.5. Kurvan rasar rejält i början – från ungefär 3500 vid ett kluster ner till runt 650 vid tre kluster, och vidare ner mot cirka 250 vid fyra kluster. Efter det planar den ut betydligt: vid fem kluster ligger inertia på ungefär 230, vid sex kluster runt 180, och ner mot cirka 110 vid nio kluster – men skillnaden per extra kluster är nu ganska liten. Figuren har till och med en pil som pekar rätt på den punkten och kallar den "Armbåge".

Jag hade valt fyra kluster. Det är precis där kurvan byter karaktär, från brant nedgång till att plana ut, vilket är den klassiska armbågen som beskrivs i avsnitt 6.2.3. Att lägga till fler kluster efter den punkten ger bara marginellt lägre inertia, så min tolkning är att fyra kluster redan fångar det mesta av strukturen i datan, och att fler kluster mest delar upp rimliga grupper i mindre bitar än nödvändigt.

Men här är det klurigt: boken visar längre fram (i avsnittet om silhouette score och silhouette diagram, figur 6.12–6.13) att datasetet faktiskt har fem sanna kluster (vilket stämmer med figur 6.5), och att fem kluster ger en mer balanserad lösning i praktiken även om både inertia och silhouette score i sig pekar mot fyra. Det tycker jag visar ganska tydligt att det inte är en exakt vetenskap att välja antal kluster:

- Armbågen i inertia-kurvan är en subjektiv bedömning – var man tycker att kurvan "knäcks" kan skilja sig åt beroende på vem som tittar, det finns inget exakt matematiskt facit.
- Olika mått (inertia, silhouette score, silhouette diagram) kan peka mot olika antal kluster, precis som här.
- I slutändan måste man väga ihop flera mått med sin egen kunskap om problemet – exempelvis om man vet att klustren "bör" vara ungefär lika stora – snarare än att mekaniskt välja det som optimerar ett enda mått.

## Fråga 5 – (Resonemangsfråga) Hur tolkar man figur 6.13 på sidan 251?

Figur 6.13 visar fyra silhouette diagram, ett för varje $k$ mellan 3 och 6. I varje diagram grupperas observationerna per kluster längs y-axeln (höjden på varje "kniv" visar hur många observationer klustret innehåller), och x-axeln visar siluettkoefficienten (ekvation 6.1) för varje observation, sorterad inom klustret – ju längre ut kniven sträcker sig, desto bättre passar punkterna in. Den streckade röda linjen är **silhouette score**, alltså medelvärdet av alla koefficienter för den modellen.

När jag går igenom de fyra delfigurerna en efter en ser jag följande: vid $k=3$ är klustret med index 1 (den mellersta kniven) mycket större än de andra två, och det har dessutom en tunn svans ner mot noll, vilket betyder att några av observationerna där ligger nära en klustergräns. Vid $k=4$ är mönstret nästan detsamma – klustret med index 1 är fortfarande klart störst. Vid $k=5$ blir klustren betydligt jämnare i storlek, även om silhouette score (den streckade linjen) faktiskt är något lägre här än vid $k=3$ och $k=4$ (se figur 6.12). Och vid $k=6$ blir resultatet sämre överlag – silhouette score sjunker ytterligare, och nu passerar inte ens alla knivarna den streckade linjen, vilket tyder på att flera observationer ligger närmare ett annat kluster än sitt eget.

Så vad väljer jag? Rent numeriskt ger $k=4$ (och $k=3$) en högre silhouette score, men jag skulle ändå gå med fem kluster, precis som boken gör, eftersom klustren då blir mer likvärdiga i storlek – vilket K-Means generellt mår bra av (jämför fråga 2). Det som är lite klurigt här är att man inte bara kan titta på det numeriska värdet på silhouette score, utan också måste bedöma hur jämna och välformade de enskilda knivarna är. Och som texten på sidan 251 påpekar hade man lika gärna kunnat välja tre eller fyra kluster – ännu en påminnelse om att det här inte är en exakt vetenskap (jämför fråga 4).

## Fråga 7 – (Koduppgift) Klustring på housing.csv

**a) Förklara vad koden gör**

Det koden gör är egentligen en enkel kundsegmentering baserad på geografi, på California housing-datasetet (`housing.csv`, samma fil som i kapitel 2). Den börjar med att importera det som behövs (`matplotlib`, `pandas`, `seaborn` och `KMeans` från scikit-learn) och läser in datan med `pd.read_csv`. Sen plockas tre variabler ut till `X`: `median_income` (medianinkomst i området), samt `latitude` och `longitude` (geografisk position), och `X.head()` skrivs ut för att man ska se hur de första raderna ser ut.

Därefter instansieras en `KMeans`-modell med `n_clusters=6`, alltså sex kluster, och `kmeans.fit_predict(X)` tränar modellen på `X` och returnerar samtidigt vilket kluster (0–5) varje observation hamnar i – resultatet läggs in som en ny kolumn `"Cluster"` i `X`. Den kolumnen görs sedan om till `category`-typ, vilket gör att seaborn tolkar den som diskret/nominal istället för en kontinuerlig skala när den används för färgläggning. `X.head()` skrivs ut igen, nu med klustertillhörigheten synlig, och till sist skapar `sns.relplot(...)` en scatterplot med longitud på x-axeln och latitud på y-axeln – i praktiken en enkel "karta" över Kalifornien – där varje punkt färgläggs efter vilket kluster den tillhör.

Resultatet blir alltså en karta där Kalifornien delas in i sex geografiska/inkomstmässiga kluster. Det som är lite lurigt är att koden inte använder `StandardScaler` innan klustringen (trots att boken rekommenderar det i avsnitt 6.2), även fast variablerna ligger på helt olika skalor – `median_income` går ungefär 0–15, medan latitud/longitud ligger mellan cirka 32–42 respektive -125 till -114. Eftersom K-Means bygger på avstånd kommer latitud och longitud, som har mycket större numerisk variation, i praktiken att dominera klustringen. Det gör att klustren mest blir geografiska regioner snarare än renodlade inkomstgrupper, vilket vi faktiskt bekräftar nedan genom att köra koden.

**b) Vilken information hade du velat ha om respektive kluster, och hur hade du designat marknadsundersökningen?**

Om jag ledde en sådan undersökning hade jag velat veta en hel del om varje kluster: demografi som ålder, hushållsstorlek, sysselsättning och utbildningsnivå, boendepreferenser som hustyp (villa, lägenhet, radhus), önskad boyta och om man föredrar nybyggt eller äldre bostäder, samt köp- och kontaktpreferenser – vilken kontaktkanal man föredrar (telefon, mejl, sociala medier, fysiska visningar), hur långt i förväg man planerar en bostadsaffär, och vilka faktorer som väger tyngst (pris, läge, skolor, pendlingsavstånd). Jag hade också velat veta något om betalningsförmåga och finansiering, alltså hur folk vanligtvis finansierar ett köp (kontant, bolån) och hur känsliga de är för ränteförändringar.

När det gäller själva undersökningen hade jag först tagit fram ett representativt urval av hushåll från varje kluster (jämför "representativa bilder" i avsnitt 6.3.2), och sedan skickat en riktad enkät eller gjort telefon-/djupintervjuer med det urvalet, med en blandning av skalfrågor och några öppna frågor. Sen hade jag jämfört svaren mellan klustren snarare än inom dem, för att se om det verkligen finns systematiska skillnader mellan de geografiska/inkomstmässiga grupperna – annars är klustringen inte särskilt användbar i praktiken. Till sist hade jag gärna kompletterat med öppna register- eller tredjepartsdata (t.ex. redan tillgänglig bostads- eller köpstatistik per område) så man slipper fråga om allt via enkät.

**c) Hur kan det här vara användbart för ett företag?**

En sån klustring kan till exempel ett fastighetsbolag eller en bank använda för att rikta marknadsföring mer träffsäkert – olika erbjudanden till olika kluster beroende på geografiskt läge och inkomstnivå (jämför kundsegmenteringsexemplet i avsnitt 6.1.1, figur 6.1). Man kan också prissätta och paketera tjänster olika, som lånevillkor, mäklartjänster eller bostadstyper anpassade efter klustret, och använda insikterna för resursallokering – var man ska öppna nya kontor, satsa säljresurser eller investera i nya bostadsprojekt, baserat på vilka områden som är mest köpstarka. Klustringen kan även användas för anomalidetektion, alltså att hitta enskilda områden som avviker kraftigt från sitt kluster, vilket kan tyda på felaktiga data eller kanske tvärtom en unik investeringsmöjlighet (jämför avsnitt 6.1.1 om anomalidetektion).

**d) Egna experiment**

Vi laddade faktiskt ner `housing.csv` och körde koden nedan, nästan oförändrad – bara sökvägen till filen justerad. Vi ville även, som boken föreslår i avsnitt 6.2.3, undersöka hur många kluster som hade varit rimligt att använda genom att titta på inertia och silhouette score.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans

df = pd.read_csv("housing.csv")
X = df.loc[:, ["median_income", "latitude", "longitude"]]
print(X.head())

kmeans = KMeans(n_clusters=6)
X["Cluster"] = kmeans.fit_predict(X)
X["Cluster"] = X["Cluster"].astype("category")
print(X.head())

sns.relplot(
    x="longitude", y="latitude", hue="Cluster", data=X, height=6,
)
plt.show()

# Lade till denna rad för att kunna redovisa resultatet numeriskt,
# eftersom vi inte kan bädda in den faktiska scatterplotten här.
print(X["Cluster"].value_counts().sort_index())

   median_income  latitude  longitude
0         8.3252     37.88    -122.23
1         8.3014     37.86    -122.22
2         7.2574     37.85    -122.24
3         5.6431     37.85    -122.25
4         3.8462     37.85    -122.25
   median_income  latitude  longitude Cluster
0         8.3252     37.88    -122.23       4
1         8.3014     37.86    -122.22       4
2         7.2574     37.85    -122.24       4
3         5.6431     37.85    -122.25       4
4         3.8462     37.85    -122.25       1
Cluster
0    7011
1    4861
2     459
3    3840
4    2786
5    1683
Name: count, dtype: int64


<Figure size 600x600 with 1 Axes>

Nedladdningen av `housing.csv` fungerade fint (samma fil som i kapitel 2), och koden kördes utan problem på totalt 20 640 rader. Eftersom `random_state` inte anges i koden varierar det exakt vilket klusterindex (0–5) som hamnar var mellan olika körningar, men storleksfördelningen brukar se ungefär likadan ut: ett par stora kluster på cirka 4 000–7 000 observationer, ett par medelstora på cirka 1 700–3 800, och ett litet kluster på bara omkring 450 observationer – troligen områdena med allra högst medianinkomst, typ dyra kustnära lägen. Scatterplotten (`sns.relplot`) ritar upp longitud mot latitud, vilket i praktiken återskapar konturerna av Kalifornien uppdelat i sex färgade regioner, precis som väntat eftersom latitud och longitud dominerar avståndsberäkningen (se resonemanget i a).

Vi kollade också på hur många kluster som hade varit rimligt genom att räkna ut inertia och silhouette score för $k = 2$ till $8$, efter att ha standardiserat variablerna med `StandardScaler` (för att ge `median_income` och de geografiska variablerna jämförbar vikt, som boken rekommenderar):

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

Xs = StandardScaler().fit_transform(
    df.loc[:, ["median_income", "latitude", "longitude"]]
)

for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    sc = silhouette_score(Xs, km.labels_)
    print(f"k={k}: inertia={km.inertia_:.1f}, silhouette={sc:.3f}")

k=2: inertia=26861.0, silhouette=0.548
k=3: inertia=18987.2, silhouette=0.518
k=4: inertia=14547.5, silhouette=0.435
k=5: inertia=11838.4, silhouette=0.387
k=6: inertia=10039.0, silhouette=0.398
k=7: inertia=8732.4, silhouette=0.370
k=8: inertia=7713.7, silhouette=0.353


Både inertia-kurvan och silhouette score pekar här mot ett litet antal kluster – silhouette score är som högst vid $k=2$ och sjunker sedan i stort sett hela vägen, med ett litet lokalt uppsving vid $k=6$. Det är samma poäng som i fråga 3 och 4 egentligen: de statistiska måtten ger ett rimligt underlag, men att man i den ursprungliga koden ändå valde $k=6$ är fullt rimligt ur ett affärsperspektiv – för en marknadsundersökning vill man ofta ha fler, mer finkorniga segment att rikta sig mot än vad ett rent statistiskt optimum ger, så länge grupperna fortfarande är tillräckligt stora och går att tolka (jämför resonemanget i fråga 4 och 5 om att antalet kluster inte är en exakt vetenskap).

## Fråga 9 – (Koduppgift) Klustringsanalys på "1980s Classic Hits with Spotify Data" (Kaggle)

Datasetet finns på Kaggle: [1980s Classic Hits with Spotify Data](https://www.kaggle.com/datasets/thebumpkin/1980s-classic-hits-with-spotify-data). Kaggle-dataset kräver ett konto och en API-nyckel (`kaggle.json`) för att laddas ner programmatiskt, vilket inte finns tillgängligt i den här miljön. Koden nedan är därför **inte körd** (`outputs: []`), men den visar en fullständig, väl underbyggd pipeline för klustringsanalysen, baserat på samma arbetsgång som i kapitlet: skala variablerna (Avsnitt 6.2), välj antal kluster med armbågsmetoden/inertia och silhouette score (Avsnitt 6.2.3), och tolka de resulterande klustren.

Datasetet innehåller låtar från 1980-talet med tillhörande Spotify "audio features" (t.ex. `danceability`, `energy`, `tempo`, `valence`, `acousticness`, `loudness`), vilka är naturliga numeriska variabler att klustra på.

In [ ]:
# Obs: kräver ett Kaggle-konto + API-nyckel (kaggle.json) för att kunna
# laddas ner, se https://www.kaggle.com/docs/api. Koden nedan är därför
# oexekverad, men visar en fullständig pipeline för klustringsanalysen.

import kaggle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1) Ladda ner och läs in datasetet
kaggle.api.dataset_download_files(
    "thebumpkin/1980s-classic-hits-with-spotify-data",
    path="data",
    unzip=True,
)
df = pd.read_csv("data/1980s_spotify_songs.csv")
print(df.head())
print(df.info())

# 2) Välj relevanta numeriska "audio features" för klustringen
features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]
X = df.loc[:, features].dropna()

# 3) Skala datan (K-Means bygger på avstånd, se Avsnitt 6.2 i boken)
X_scaled = StandardScaler().fit_transform(X)

# 4) Välj antal kluster med armbågsmetoden (inertia) och silhouette score
inertias = []
sil_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("Antalet kluster")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Armbågsmetoden")

axes[1].plot(list(k_range), sil_scores, marker="o", color="tab:orange")
axes[1].set_xlabel("Antalet kluster")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette score per antal kluster")
plt.show()

# 5) Träna den slutgiltiga modellen med det valda antalet kluster
#    (antag t.ex. att armbågen/högst silhouette score pekar på k=4)
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df.loc[X.index, "Cluster"] = kmeans.fit_predict(X_scaled)
df["Cluster"] = df["Cluster"].astype("category")

# 6) Tolka klustren: jämför medelvärdet av varje audio feature per kluster
cluster_profile = df.groupby("Cluster")[features].mean()
print(cluster_profile)

# 7) Visualisera t.ex. energy mot danceability, färgat efter kluster
sns.relplot(
    x="danceability", y="energy", hue="Cluster", data=df, height=6,
)
plt.show()

**Tolkning av klustren:** när modellen är tränad hade jag kikat på `cluster_profile`-tabellen (medelvärdet av varje audio feature per kluster) för att försöka ge varje kluster en musikalisk profil. Man kan tänka sig till exempel ett kluster med hög `danceability`, `energy` och `valence` samt snabbt `tempo` – glada discolåtar och dansvänliga hits – ett annat med hög `acousticness` och låg `energy`, alltså mer akustiska ballader, och ett tredje med hög `loudness`/`energy` men lägre `valence`, som skulle bli de rockigare, mer intensiva låtarna med mindre "glad" karaktär. Precis som i fråga 3–5 hade valet av exakt antal kluster ($k$) krävt en kombination av armbågsmetoden, silhouette score och eget musikaliskt omdöme, snarare än att det finns ett enda rätt svar.